In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import MinMaxScaler
from skimage.metrics import structural_similarity as ssim
from math import log10
import json
import os
import time
import json
import os

from sklearn.metrics import mean_squared_error
from processdata import TimeSeriesDataset

In [ ]:
# Define os caminhos de salvamento e carregamento
# save_path = r"./results/csshred/testes"
# load_path = r"./results/csshred/testes"
save_path = r"/home/romulo/Documentos/lpips-env/results/csshred/oldroyd_paper"
load_path = r"/home/romulo/Documentos/lpips-env/results/csshred/oldroyd_paper"

# Carrega os arquivos necessários
test_recons = np.load(load_path + r"/test_recons.npy")
test_ground_truth = np.load(load_path + r"/test_ground_truth.npy")
matrix = np.load(load_path + r"/matrix.npy")
snapshot = np.load(load_path + r"/snapshot.npy")
sensor_positions_x = np.load(load_path + r"/sensor_positions_x.npy")
sensor_positions_y = np.load(load_path + r"/sensor_positions_y.npy")
train_error = np.load(load_path + r"/train_error.npy")
validation_errors = np.load(load_path + r"/validation_errors.npy")

## GERAR IMAGENS CS-SHRED

In [ ]:
def plot_results(
    train_error, validation_errors, test_recons, test_ground_truth, matrix, subsampled,
    save_path=r'./results/csshred/oldroyd/'
):
    os.makedirs(save_path, exist_ok=True)
    
    dim_x, dim_y, dim_t = subsampled.shape

    test_ground_truth = test_ground_truth.copy().reshape(-1, dim_x, dim_y)
    test_ground_truth = np.transpose(test_ground_truth.copy(), (1, 2, 0))
   
    test_recons = test_recons.reshape(-1, dim_x, dim_y)
   
    subsampled = subsampled.reshape(dim_x, dim_y, -1)

    print("test_recons", test_recons.shape)
    print("test_ground_truth", test_ground_truth.shape)
    print("subsampled", subsampled.shape)

    # Plot Training Error
    plt.figure(figsize=(12, 6))
    plt.plot(train_error, label="Training Error")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Error Over Epochs")
    plt.legend()
    training_error_plot_path = os.path.join(save_path, "training_error.pdf")
    plt.savefig(training_error_plot_path)
    plt.show()

    # Plot Validation Error
    plt.figure(figsize=(12, 6))
    plt.plot(validation_errors, label="Validation Error")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Validation Error Over Epochs")
    plt.legend()
    validation_error_plot_path = os.path.join(save_path, "validation_error.pdf")
    plt.savefig(validation_error_plot_path)
    plt.show()

    # Plot Last Snapshot - Reconstructed and Original
    plt.figure(figsize=(14, 6))

    ax1 = plt.subplot(1, 2, 1)
    img1 = ax1.imshow(test_recons[-1, :, :], cmap="viridis", origin="lower", extent=[0, 1, 0, 1])
    fig1 = plt.gcf()
    cbar1 = fig1.colorbar(img1, ax=ax1, label=r"$tr(C)$")
    plt.title("Reconstructed - Last Snapshot")
    
    ax2 = plt.subplot(1, 2, 2)
    img2 = ax2.imshow(test_ground_truth[:, :, -1], cmap="viridis", origin="lower", extent=[0, 1, 0, 1])
    fig2 = plt.gcf()
    cbar2 = fig2.colorbar(img2, ax=ax2, label=r"$tr(C)$")
    plt.title("Original - Last Snapshot")
    
    last_snapshot_plot_path = os.path.join(save_path, "last_snapshot_comparison.pdf")
    plt.savefig(last_snapshot_plot_path)
    plt.show()

    # Plot Last Snapshot - Reconstructed and Subsampled
    plt.figure(figsize=(14, 6))
    
    ax3 = plt.subplot(1, 2, 1)
    img3 = ax3.imshow(test_recons[-1, :, :], cmap="viridis", origin="lower", extent=[0, 1, 0, 1])
    fig3 = plt.gcf()
    cbar3 = fig3.colorbar(img3, ax=ax3, label=r"$tr(C)$")
    plt.title("Reconstructed - Last Snapshot")
    
    ax4 = plt.subplot(1, 2, 2)
    img4 = ax4.imshow(subsampled[:, :, -1], cmap="viridis", origin="lower", extent=[0, 1, 0, 1])
    fig4 = plt.gcf()
    cbar4 = fig4.colorbar(img4, ax=ax4, label=r"$tr(C)$")
    plt.title("Subsampled - Last Snapshot")
    
    subsampled_snapshot_plot_path = os.path.join(save_path, "subsampled_snapshot_comparison.pdf")
    plt.savefig(subsampled_snapshot_plot_path)
    plt.show()
    
    # Calculando o SSIM, MSE e PSNR para o último snapshot
     # Calculando métricas
    ssim_value_last = ssim(
        test_ground_truth[:, :, -1],
        test_recons[-1],
        data_range=test_ground_truth[:, :, -1].max() - test_ground_truth[:, :, -1].min(),
    )
    
    test_recons_mse = np.transpose(test_recons.copy(), (1,2,0))
    mse_value_last = np.linalg.norm(test_recons_mse - test_ground_truth) / np.linalg.norm(test_ground_truth)
    
    mse = mean_squared_error(test_ground_truth[:, :, -1], test_recons[-1])
    max_pixel = np.max(test_ground_truth[:, :, -1])
    psnr_value_last = 20 * log10(max_pixel / np.sqrt(mse))
    
    # Cálculo do erro normalizado
    error_norm_last = np.linalg.norm(test_recons[-1] - test_ground_truth[:, :, -1]) / np.linalg.norm(test_ground_truth[:, :, -1])
    
    print(f"SSIM for the last snapshot: {ssim_value_last}")
    print(f"MSE for the last snapshot: {mse_value_last}")
    print(f"PSNR for the last snapshot: {psnr_value_last} dB")
    print(f"Normalized Error for the last snapshot: {error_norm_last}")

    # Calculando médias para todos os snapshots
    ssim_values = []
    psnr_values = []
    error_norms = []  # Lista para armazenar os erros normalizados
    for i in range(test_recons.shape[0]):
        ssim_value = ssim(
            test_ground_truth[:, :, i],
            test_recons[i],
            data_range=test_ground_truth[:, :, i].max() - test_ground_truth[:, :, i].min(),
        )
        ssim_values.append(ssim_value)
        
        mse = mean_squared_error(test_ground_truth[:, :, i], test_recons[i])
        max_pixel = np.max(test_ground_truth[:, :, i])
        psnr = 20 * log10(max_pixel / np.sqrt(mse))
        psnr_values.append(psnr)

        # Cálculo do erro normalizado para cada snapshot
        error_norm = np.linalg.norm(test_recons[i] - test_ground_truth[:, :, i]) / np.linalg.norm(test_ground_truth[:, :, i])
        error_norms.append(error_norm)

    mean_ssim_value = np.mean(ssim_values)
    mean_psnr_value = np.mean(psnr_values)
    mean_error_norm = np.mean(error_norms)  
    print(f"Mean SSIM for all snapshots: {mean_ssim_value}")
    print(f"Mean PSNR for all snapshots: {mean_psnr_value} dB")
    print(f"Mean Normalized Error for all snapshots: {mean_error_norm}")

    # Salvando resultados em JSON
    results = {
        "SSIM_last_snapshot": float(ssim_value_last),
        "MSE_last_snapshot": float(mse_value_last),
        "PSNR_last_snapshot": float(psnr_value_last),
        "Normalized_Error_last_snapshot": float(error_norm_last),  
        "Mean_SSIM_all_snapshots": float(mean_ssim_value),
        "Mean_PSNR_all_snapshots": float(mean_psnr_value),
        "Mean_Normalized_Error_all_snapshots": float(mean_error_norm), 
    }
    
    json_file_path = os.path.join(save_path, "results.json")
    with open(json_file_path, "w") as json_file:
        json.dump(results, json_file, indent=4)
    
    print(f"Results saved to {json_file_path}")
    print(f"Plots saved to {save_path}")

In [ ]:
plot_results(
    train_error, validation_errors, test_recons, test_ground_truth, matrix, snapshot,
    save_path=save_path
)

## GERAR IMAGENS SHRED

In [ ]:
# Define os caminhos de salvamento e carregamento
# save_path = r"./results/csshred/testes"
# load_path = r"./results/csshred/testes"
save_path = r"/home/romulo/Documentos/lpips-env/results/shred/oldroyd"
load_path = r"/home/romulo/Documentos/lpips-env/results/shred/oldroyd"

# Carrega os arquivos necessários
test_recons = np.load(load_path + r"/test_recons.npy")
test_ground_truth = np.load(load_path + r"/test_ground_truth.npy")
matrix = np.load(load_path + r"/matrix.npy")
snapshot = np.load(load_path + r"/snapshot.npy")
sensor_positions_x = np.load(load_path + r"/sensor_positions_x.npy")
sensor_positions_y = np.load(load_path + r"/sensor_positions_y.npy")
# train_error = np.load(load_path + r"/train_error.npy")
validation_errors = np.load(load_path + r"/validation_errors.npy")

In [ ]:

def plot_results(
    validation_errors, test_recons, test_ground_truth, matrix, subsampled,
    save_path=r'./results/csshred/oldroyd/'
):
    os.makedirs(save_path, exist_ok=True)
    
    dim_x, dim_y, dim_t = subsampled.shape

    test_ground_truth = test_ground_truth.copy().reshape(-1, dim_x, dim_y)
    test_ground_truth = np.transpose(test_ground_truth.copy(), (1, 2, 0))
   
    test_recons = test_recons.reshape(-1, dim_x, dim_y)
   
    subsampled = subsampled.reshape(dim_x, dim_y, -1)

    print("test_recons", test_recons.shape)
    print("test_ground_truth", test_ground_truth.shape)
    print("subsampled", subsampled.shape)

    # Plot Training Error
    # plt.figure(figsize=(12, 6))
    # plt.plot(train_error, label="Training Error")
    # plt.xlabel("Epoch")
    # plt.ylabel("Loss")
    # plt.title("Training Error Over Epochs")
    # plt.legend()
    # training_error_plot_path = os.path.join(save_path, "training_error.png")
    # plt.savefig(training_error_plot_path)
    # plt.show()

    # Plot Validation Error
    plt.figure(figsize=(12, 6))
    plt.plot(validation_errors, label="Validation Error")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Validation Error Over Epochs")
    plt.legend()
    validation_error_plot_path = os.path.join(save_path, "validation_error.png")
    plt.savefig(validation_error_plot_path)
    plt.show()

    # Plot Last Snapshot - Reconstructed and Original
    plt.figure(figsize=(14, 6))

    ax1 = plt.subplot(1, 2, 1)
    img1 = ax1.imshow(test_recons[-1, :, :], cmap="viridis", origin="lower")
    fig1 = plt.gcf()
    cbar1 = fig1.colorbar(img1, ax=ax1, label=r"$tr(C)$")
    plt.title("Reconstructed - Last Snapshot")
    
    ax2 = plt.subplot(1, 2, 2)
    img2 = ax2.imshow(test_ground_truth[:, :, -1], cmap="viridis", origin="lower")
    fig2 = plt.gcf()
    cbar2 = fig2.colorbar(img2, ax=ax2, label=r"$tr(C)$")
    plt.title("Original - Last Snapshot")
    
    last_snapshot_plot_path = os.path.join(save_path, "last_snapshot_comparison.png")
    plt.savefig(last_snapshot_plot_path)
    plt.show()

    # Plot Last Snapshot - Reconstructed and Subsampled
    plt.figure(figsize=(14, 6))
    
    ax3 = plt.subplot(1, 2, 1)
    img3 = ax3.imshow(test_recons[-1, :, :], cmap="viridis", origin="lower")
    fig3 = plt.gcf()
    cbar3 = fig3.colorbar(img3, ax=ax3, label=r"$tr(C)$")
    plt.title("Reconstructed - Last Snapshot")
    
    ax4 = plt.subplot(1, 2, 2)
    img4 = ax4.imshow(subsampled[:, :, -1], cmap="viridis", origin="lower")
    fig4 = plt.gcf()
    cbar4 = fig4.colorbar(img4, ax=ax4, label=r"$tr(C)$")
    plt.title("Subsampled - Last Snapshot")
    
    subsampled_snapshot_plot_path = os.path.join(save_path, "subsampled_snapshot_comparison.png")
    plt.savefig(subsampled_snapshot_plot_path)
    plt.show()
    
    # Calculando o SSIM, MSE e PSNR para o último snapshot
     # Calculando métricas
    ssim_value_last = ssim(
        test_ground_truth[:, :, -1],
        test_recons[-1],
        data_range=test_ground_truth[:, :, -1].max() - test_ground_truth[:, :, -1].min(),
    )
    
    test_recons_mse = np.transpose(test_recons.copy(), (1,2,0))
    mse_value_last = np.linalg.norm(test_recons_mse - test_ground_truth) / np.linalg.norm(test_ground_truth)
    
    mse = mean_squared_error(test_ground_truth[:, :, -1], test_recons[-1])
    max_pixel = np.max(test_ground_truth[:, :, -1])
    psnr_value_last = 20 * log10(max_pixel / np.sqrt(mse))
    
    # Cálculo do erro normalizado
    error_norm_last = np.linalg.norm(test_recons[-1] - test_ground_truth[:, :, -1]) / np.linalg.norm(test_ground_truth[:, :, -1])
    
    print(f"SSIM for the last snapshot: {ssim_value_last}")
    print(f"MSE for the last snapshot: {mse_value_last}")
    print(f"PSNR for the last snapshot: {psnr_value_last} dB")
    print(f"Normalized Error for the last snapshot: {error_norm_last}")

    # Calculando médias para todos os snapshots
    ssim_values = []
    psnr_values = []
    error_norms = []  # Lista para armazenar os erros normalizados
    for i in range(test_recons.shape[0]):
        ssim_value = ssim(
            test_ground_truth[:, :, i],
            test_recons[i],
            data_range=test_ground_truth[:, :, i].max() - test_ground_truth[:, :, i].min(),
        )
        ssim_values.append(ssim_value)
        
        mse = mean_squared_error(test_ground_truth[:, :, i], test_recons[i])
        max_pixel = np.max(test_ground_truth[:, :, i])
        psnr = 20 * log10(max_pixel / np.sqrt(mse))
        psnr_values.append(psnr)

        # Cálculo do erro normalizado para cada snapshot
        error_norm = np.linalg.norm(test_recons[i] - test_ground_truth[:, :, i]) / np.linalg.norm(test_ground_truth[:, :, i])
        error_norms.append(error_norm)

    mean_ssim_value = np.mean(ssim_values)
    mean_psnr_value = np.mean(psnr_values)
    mean_error_norm = np.mean(error_norms)  # Média dos erros normalizados
    print(f"Mean SSIM for all snapshots: {mean_ssim_value}")
    print(f"Mean PSNR for all snapshots: {mean_psnr_value} dB")
    print(f"Mean Normalized Error for all snapshots: {mean_error_norm}")

    # Salvando resultados em JSON
    results = {
        "SSIM_last_snapshot": float(ssim_value_last),
        "MSE_last_snapshot": float(mse_value_last),
        "PSNR_last_snapshot": float(psnr_value_last),
        "Normalized_Error_last_snapshot": float(error_norm_last),  
        "Mean_SSIM_all_snapshots": float(mean_ssim_value),
        "Mean_PSNR_all_snapshots": float(mean_psnr_value),
        "Mean_Normalized_Error_all_snapshots": float(mean_error_norm),  
    }
    
    json_file_path = os.path.join(save_path, "results.json")
    with open(json_file_path, "w") as json_file:
        json.dump(results, json_file, indent=4)
    
    print(f"Results saved to {json_file_path}")
    print(f"Plots saved to {save_path}")